# Recursive Latent Reasoner (RLR) — All Gates Colab Runner

This notebook executes the complete verification suite from the **Corrected Build Plan (v2)** across **Gate G1**, **Gate G2**, and **Gate G3** on an NVIDIA **A100 GPU** with High-RAM.

- **Gate G1 (Track 1 — RLR)**: Evaluates the 5-probe cheat detection suite (prompt shuffle, loop collapse, max_loops=1 lobotomy, semantic shift, OOD hops 9–12) and scratchpad utilization gate ($\Delta \ge +10\,\text{pp}$).
- **Gate G2 (Track 2 — SSM-MBRL)**: Compares `Mamba2WorldModel` vs parameter-matched `GRURSSMWorldModel` on POPGym memory tasks across seeds.
- **Gate G3 (Bridge Gate)**: Runs the 3-arm ablation grid (`full` vs `frozen` vs `absent`) across loop budgets and ponder costs $\tau$.


## ⚡ Quick-Start: 1-Click Unattended Run (Option B — ~1.5h on A100)

Run the cell below, grant Google Drive permissions, and you can leave it unattended.
- Mounts Google Drive (`/content/drive/MyDrive/RLR_Runs/`)
- Unpacks bundle
- Runs all 3 gates in **RAPID mode** (Option B: 3 seeds, ~1.5 hours)
- Automatically saves `Master_Gate_Report.md` & `pipeline.log` to Drive
- **Automatically disconnects VM (`runtime.unassign()`)** when done to save A100 compute units!


In [ ]:
# 1. Mount Google Drive for persistent logging
from google.colab import drive
drive.mount("/content/drive")

# 2. Clone repository directly from GitHub (Zero manual file uploads!)
import os
if not os.path.exists("/content/Mamba2-Recursive-Latent-Forcing-Operates"):
    !git clone https://github.com/vitorcalvi/Mamba2-Recursive-Latent-Forcing-Operates.git /content/Mamba2-Recursive-Latent-Forcing-Operates

%cd /content/Mamba2-Recursive-Latent-Forcing-Operates
!git pull origin main

# 3. Run all gates unattended in Option B (Rapid Verification Grid: 3 seeds, ~1.5h on A100)
!python3 run_colab_unattended.py --mode rapid --output-dir /content/drive/MyDrive/RLR_Runs


## 1. Hardware & Environment Check
Ensure **A100 GPU** (Ampere `sm_80`) is assigned.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {device_name} ({total_vram:.2f} GB VRAM, sm_{cap[0]}{cap[1]})")
    assert cap[0] >= 8, "Recommended GPU is A100 (sm_80) or L4 (sm_89) for Ampere/Ada tensor cores."


## 2. Dependency Installation
Installs core dependencies. On A100, pre-built `causal-conv1d` and `mamba-ssm` wheels install cleanly; pure-PyTorch fallbacks remain active as safety nets.

In [ ]:
!pip install -q einops sentencepiece gymnasium popgym transformers

# Optional compiled Mamba-2 kernels for Ampere A100:
!pip install -q causal-conv1d>=1.4.0 mamba-ssm>=2.2.0 --no-build-isolation || echo "Using pure-PyTorch scan fallback"


## 3. Workspace / Repository Setup
Sets up python path and verifies package imports.

In [ ]:
import os, sys

# If running in Google Colab, unpack bundle if present
if os.path.exists("/content/rlr_colab_bundle.zip") and not os.path.exists("/content/rlr"):
    !unzip -q /content/rlr_colab_bundle.zip -d /content/
    %cd /content
elif os.path.exists("rlr_colab_bundle.zip") and not os.path.exists("rlr"):
    !unzip -q rlr_colab_bundle.zip

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import rlr
import t2
import bridge

print(f"rlr modules: {rlr.__all__}")
print(f"t2 MAMBA_AVAILABLE: {t2.MAMBA_AVAILABLE}")
print(f"t2 POPGYM_AVAILABLE: {t2.POPGYM_AVAILABLE}")
print(f"bridge modules: {bridge.__all__}")
print("All packages imported cleanly!")


## 4. Gate G1: Track 1 (RLR) Anti-Cheat & Reasoner Validation

Runs all 6 anti-cheat checks:
1. Multi-hop chain accuracy $\ge 60\%$
2. Scratchpad ablation $\Delta \ge +10.0\,\text{pp}$
3. Prompt-shuffle degradation $\le 15\%$
4. No loop collapse (confidence stdev $\ge 0.05$)
5. Loop removal lobotomy drop $\ge 10\%$
6. OOD hops 9–12 extrapolation $> 0\%$


In [ ]:
from eval.gate_g1 import GateG1
from IPython.display import Markdown, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running Gate G1 on {device}...")

gate_g1 = GateG1()
result_g1 = gate_g1.evaluate(device=device)

display(Markdown(result_g1.summary))
print(f"Gate G1 Passed: {result_g1.passed}")


## 5. Gate G2: Track 2 (SSM-MBRL) vs GRU-RSSM Comparison Grid

Evaluates `Mamba2WorldModel` vs `GRURSSMWorldModel` on POPGym memory benchmarks across seeds at **identical parameter count (113,674 params)**.

In [ ]:
from eval.gate_g2 import GateG2
from IPython.display import Markdown, display

print("Running Gate G2 (POPGym Benchmark Grid across seeds)...")

# Use fast_mode=False for the full 5-seed grid, or fast_mode=True for quick smoke test
gate_g2 = GateG2(n_seeds=5)
result_g2 = gate_g2.evaluate(fast_mode=False)

display(Markdown(result_g2.summary))
print(f"Gate G2 Passed: {result_g2.passed}")


## 6. Gate G3: Track 1 ↔ Track 2 Bridge 3-Arm Ablation Grid

Evaluates the conditional coupling of RLR latent looping to DreamerV3 world-model policy:
- **FULL**: Active recursive scratchpad with ACT halting
- **FROZEN**: Fixed-weight random recurrence control
- **ABSENT**: Standard un-augmented policy

Requires $\ge +15\%$ score improvement over absent baseline at equal wall-clock.

In [ ]:
from eval.gate_g3 import GateG3
from IPython.display import Markdown, display

print("Running Gate G3 (3-Arm Bridge Ablation Grid)...")

gate_g3 = GateG3(env_name="RepeatPrevious", n_seeds=5, min_improvement_pct=0.15)
result_g3 = gate_g3.evaluate(fast_mode=False)

display(Markdown(result_g3.summary))
print(f"Gate G3 Passed: {result_g3.passed}")


## 7. Master Executive Dashboard
Consolidated summary across all three gates.

In [ ]:
dashboard = f"""# Recursive Latent Reasoner (RLR) — Master Gate Status

| Gate | Domain | Objective | Status |
|:---|:---|:---|:---|
| **Gate G1** | Track 1 (RLR) | Anti-Cheat & Reasoner Validation | {"✅ PASSED" if result_g1.passed else "🛑 FAILED"} |
| **Gate G2** | Track 2 (SSM-MBRL) | Mamba-2 vs GRU-RSSM Parity/Win (POPGym) | {"✅ PASSED" if result_g2.passed else "🛑 FAILED"} |
| **Gate G3** | T1↔T2 Bridge | +15% Hard-Mode Gain over Baseline | {"✅ PASSED" if result_g3.passed else "🛑 FAILED"} |

**Decision Protocol:**
- If G1 and G2 pass: Proceed to End-to-End Bridge RL experiments.
- If G1 fails: Archive T1 as an O(1)-VRAM looping mechanism, not an algorithmic reasoner.
- If G2 fails: Archive SSM world models as inferior to standard GRU-RSSM on memory credit tasks.
- If G3 fails: Document negative result ablation table per empirical research standards.
"""

display(Markdown(dashboard))
